In [578]:
import pandas as pd

In [579]:
df = pd.read_csv('../data/retail_store_sales.csv')

## Init info

In [580]:
df.head(3)

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False


In [581]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  str    
 1   Customer ID       12575 non-null  str    
 2   Category          12575 non-null  str    
 3   Item              11362 non-null  str    
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  str    
 8   Location          12575 non-null  str    
 9   Transaction Date  12575 non-null  str    
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(1), str(7)
memory usage: 1.9+ MB


## NaN's amounts

In [582]:
df.isna().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [583]:
df.duplicated().sum()

np.int64(0)

## Duplicates' amounts

In [584]:
df['Transaction ID'].duplicated().sum()

np.int64(0)

## Comvert Date object type to datetime

In [585]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])
df['Transaction Date'] 

0       2024-04-08
1       2023-07-23
2       2022-10-05
3       2022-05-07
4       2022-10-02
           ...    
12570   2023-09-03
12571   2022-08-12
12572   2024-08-24
12573   2023-12-30
12574   2022-08-06
Name: Transaction Date, Length: 12575, dtype: datetime64[us]

## Handle price NaNs

In [586]:
complete = df.dropna(subset=['Price Per Unit', 'Quantity', 'Total Spent'])
check = (complete['Price Per Unit'] * complete['Quantity'] == complete['Total Spent'])
complete[~check].shape

(0, 11)

In [587]:
df[['Price Per Unit', 'Quantity', 'Total Spent']].isna().sum()

Price Per Unit    609
Quantity          604
Total Spent       604
dtype: int64

In [588]:
trio = df[['Price Per Unit', 'Quantity', 'Total Spent']]

In [589]:
price_nans_per_row = df[['Price Per Unit', 'Quantity', 'Total Spent']].isna().sum(axis=1)

In [590]:
price_nans_per_row.value_counts().sort_index()

0    11362
1      609
2      604
Name: count, dtype: int64

In [591]:
trio[price_nans_per_row == 1].isna().sum()

Price Per Unit    609
Quantity            0
Total Spent         0
dtype: int64

In [592]:
trio[price_nans_per_row == 2].isna().sum()

Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64

In [593]:
mask = df['Price Per Unit'].isna() & df['Quantity'].notna() & df['Total Spent'].notna()

In [594]:
mask.value_counts()

False    11966
True       609
Name: count, dtype: int64

In [595]:
df.loc[mask, 'Price Per Unit'] = df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']

In [596]:
df[['Price Per Unit', 'Quantity', 'Total Spent']].isna().sum()

Price Per Unit      0
Quantity          604
Total Spent       604
dtype: int64

In [597]:
df['Category'].value_counts()

Category
Furniture                             1591
Electric household essentials         1591
Food                                  1588
Milk Products                         1584
Butchers                              1568
Beverages                             1567
Computers and electric accessories    1558
Patisserie                            1528
Name: count, dtype: int64

In [598]:
cat_medians = df.groupby('Category')['Quantity'].median()
cat_medians

Category
Beverages                             6.0
Butchers                              5.0
Computers and electric accessories    6.0
Electric household essentials         6.0
Food                                  6.0
Furniture                             6.0
Milk Products                         6.0
Patisserie                            6.0
Name: Quantity, dtype: float64

In [599]:
still_missing = df['Quantity'].isna() & df['Total Spent'].isna()
still_missing

0        False
1        False
2        False
3        False
4        False
         ...  
12570    False
12571    False
12572    False
12573    False
12574    False
Length: 12575, dtype: bool

In [600]:
df.isna().sum()

Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit         0
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64

In [601]:
df['Quantity Estimated'] = still_missing
df[still_missing]

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied,Quantity Estimated
7,TXN_1372952,CUST_21,Furniture,NaN,33.5,NaN,NaN,Digital Wallet,In-store,2024-04-02,True,True
15,TXN_1809665,CUST_14,Beverages,NaN,24.5,NaN,NaN,Credit Card,In-store,2022-05-11,NaN,True
19,TXN_4206593,CUST_01,Furniture,NaN,35.0,NaN,NaN,Digital Wallet,Online,2025-01-13,False,True
25,TXN_3481599,CUST_05,Furniture,NaN,39.5,NaN,NaN,Cash,Online,2022-09-08,False,True
34,TXN_1621497,CUST_06,Patisserie,NaN,23.0,NaN,NaN,Cash,In-store,2023-02-18,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...
12527,TXN_1069238,CUST_23,Food,NaN,5.0,NaN,NaN,Digital Wallet,In-store,2022-08-13,False,True
12552,TXN_4823896,CUST_05,Milk Products,NaN,8.0,NaN,NaN,Cash,In-store,2022-07-21,False,True
12556,TXN_4397672,CUST_04,Beverages,NaN,41.0,NaN,NaN,Credit Card,Online,2024-11-28,True,True
12562,TXN_7422454,CUST_07,Butchers,NaN,33.5,NaN,NaN,Cash,Online,2023-04-15,NaN,True


In [ ]:
df.loc[still_missing, 'Quantity'] = df.loc[still_missing, 'Category'].map(cat_medians)

In [ ]:
df.loc[still_missing, 'Total Spent'] = df.loc[still_missing, 'Quantity'] * df.loc[still_missing, 'Price Per Unit']

In [ ]:
df[['Price Per Unit', 'Quantity', 'Total Spent']].isna().sum()

Price Per Unit    0
Quantity          0
Total Spent       0
dtype: int64

## Handle 'Item' NaNs

In [611]:
df.groupby('Category')['Item'].apply(lambda x: x.isna().mean()).round(3)

Category
Beverages                             0.089
Butchers                              0.094
Computers and electric accessories    0.103
Electric household essentials         0.097
Food                                  0.102
Furniture                             0.082
Milk Products                         0.100
Patisserie                            0.104
Name: Item, dtype: float64

In [612]:
df.groupby('Location')['Item'].apply(lambda x: x.isna().mean()).round(3)

Location
In-store    0.097
Online      0.096
Name: Item, dtype: float64

In [613]:
df['Item'] = df['Item'].fillna('Unknown')
df['Item'].isna().sum()

np.int64(0)

In [614]:
df['Item'].value_counts()

Item
Unknown         1213
Item_2_BEV       126
Item_25_FUR      113
Item_11_FUR      110
Item_16_MILK     109
                ... 
Item_13_BEV        7
Item_5_BEV         7
Item_13_FUR        7
Item_21_PAT        6
Item_3_EHE         5
Name: count, Length: 201, dtype: int64

In [615]:
df.isna().sum()

Transaction ID           0
Customer ID              0
Category                 0
Item                     0
Price Per Unit           0
Quantity                 0
Total Spent              0
Payment Method           0
Location                 0
Transaction Date         0
Discount Applied      4199
Quantity Estimated       0
dtype: int64

## Handle 'Discount Applied' NaNs

In [616]:
df['Discount Applied'].value_counts()

Discount Applied
True     4219
False    4157
Name: count, dtype: int64

In [617]:
df.groupby(df['Transaction Date'].dt.year)['Discount Applied'].apply(lambda x: x.isna().mean()).round(3)

Transaction Date
2022    0.323
2023    0.347
2024    0.331
2025    0.357
Name: Discount Applied, dtype: float64

In [618]:
df.groupby('Location')['Discount Applied'].apply(lambda x: x.isna().mean()).round(3)

Location
In-store    0.337
Online      0.331
Name: Discount Applied, dtype: float64

In [619]:
df.groupby('Payment Method')['Discount Applied'].apply(lambda x: x.isna().mean()).round(3)

Payment Method
Cash              0.342
Credit Card       0.331
Digital Wallet    0.328
Name: Discount Applied, dtype: float64

In [620]:
df['Discount Applied'] = df['Discount Applied'].map({True: 'Yes', False: 'No'}).fillna('Unknown')
df['Discount Applied'].value_counts()

Discount Applied
Yes        4219
Unknown    4199
No         4157
Name: count, dtype: int64

In [621]:
df.isna().sum()

Transaction ID        0
Customer ID           0
Category              0
Item                  0
Price Per Unit        0
Quantity              0
Total Spent           0
Payment Method        0
Location              0
Transaction Date      0
Discount Applied      0
Quantity Estimated    0
dtype: int64

In [622]:
df.to_csv('../output/retail_store_sales_cleaned.csv', index=False)